# Baseline HTTP evaluation notebook
Run the baseline FastAPI service and summarize p50/p95 latency, throughput, error rate, and tested concurrency.


In [ ]:
import os, json, time, subprocess, requests, pathlib, signal
BASE_URL = os.getenv('BASE_URL', 'http://baseline_http:8000')
RESULT_DIR = pathlib.Path('results')
RESULT_DIR.mkdir(exist_ok=True)
print({'BASE_URL': BASE_URL, 'RESULT_DIR': str(RESULT_DIR)})


## 1. Wait for the service to become ready


In [ ]:
for _ in range(60):
    try:
        r = requests.get(f'{BASE_URL}/readyz', timeout=3)
        if r.status_code == 200:
            print(r.json())
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError('baseline_http service did not become ready')


## 2. Validate the agreed sample request/response contract


In [ ]:
payload = {
    'transaction_description': 'STARBUCKS STORE 1458 NEW YORK NY',
    'country': 'US',
    'currency': 'USD'
}
resp = requests.post(f'{BASE_URL}/predict', json=payload, timeout=5)
print(resp.status_code)
print(resp.json())


## 3. Run HTTP benchmark summary


In [ ]:
subprocess.run([
    'python',
    'scripts/benchmark_http.py'
], check=True)


## 4. Inspect saved summary


In [ ]:
import pandas as pd
df = pd.read_csv('results/baseline_http_summary.csv')
df
